In [1]:
import numpy as np
import pickle
import os
from sklearn import preprocessing

In [23]:
var_names = ["ht", "met", "m_jj", "tau21_j1", "tau21_j2", "tau32_j1", "tau32_j2"]

mc_path = "data/chunks_MC"
data_path = "data/chunks"

research_proj = "SemiVisJets"

# Chunk 01 will be used for test
# Chunk 02 - 05 will be used for training

In [29]:
print("Loading Test MC data...")
data_test = np.load(f"{mc_path}/mc_chunk_01.npz", allow_pickle=True)
test_mc_event = np.column_stack([data_test[var] for var in var_names])

print("Checking MC test data...")
print(test_mc_event.shape)

Loading Test MC data...
Checking MC test data...
(10000000, 7)


In [17]:
print("Loading MC data... Chunk 02 - 05")

mc_events = {}
for i in range(2, 6):
    data = np.load(f"{mc_path}/mc_chunk_{i:02d}.npz",  allow_pickle=True)
    # print(data.files)
    # Stack all arrays column-wise into one array per chunk
    mc_events[i] = np.column_stack([data[k] for k in var_names])

print("Checking MC data shapes...")
for i in range(2, 6):
    print(f"Chunk {i:02d}: {mc_events[i].shape}") 

Loading MC data... Chunk 02 - 05
Checking MC data shapes...
Chunk 02: (10000000, 7)
Chunk 03: (10000000, 7)
Chunk 04: (10000000, 7)
Chunk 05: (5520132, 7)


In [ ]:
print("Scaling MC data...")
mc_scaler_path = "SemiVisJets/mc_scalers"
mc_scaler = {}
for i in range(2, 6):
    scaler = preprocessing.MinMaxScaler(feature_range=(-2.5, 2.5)).fit(mc_events[i])
    with open(f"{mc_scaler_path}/mc_scaler_chunk{i:02d}.pkl","wb") as f:
        print("Saving out trained minmax scaler.")
        pickle.dump(scaler, f)
    mc_scaler[i] = scaler

Scaling MC data...
Saving out trained minmax scaler.
Saving out trained minmax scaler.
Saving out trained minmax scaler.
Saving out trained minmax scaler.


In [28]:
# SR Cuts

def sr_mask(events):
    ht = events[:,0]
    met = events[:,1]
    # Apply SR cuts
    mask = (ht > 600) & (met > 600)
    return mask



for i in range(2, 6):
    print("Before cuts: ", mc_events[i].shape)
    mc_mask_SR = sr_mask(mc_events[i])
    mc_mask_CR = ~mc_mask_SR
    print(f"Chunk {i:02d}: SR events: {np.sum(mc_mask_SR)}")
    print(f"Chunk {i:02d}: CR events: {np.sum(mc_mask_CR)}")

    np.savez(f"{research_proj}/data/mc_events_chunk{i:02d}.npz", mc_events_cr=mc_scaler[i].transform(mc_events[i][mc_mask_CR]), mc_events_sr=mc_scaler[i].transform(mc_events[i][mc_mask_SR]))

Before cuts:  (10000000, 7)
Chunk 02: SR events: 47151
Chunk 02: CR events: 9952849
Before cuts:  (10000000, 7)
Chunk 03: SR events: 46968
Chunk 03: CR events: 9953032
Before cuts:  (10000000, 7)
Chunk 04: SR events: 47018
Chunk 04: CR events: 9952982
Before cuts:  (5520132, 7)
Chunk 05: SR events: 26184
Chunk 05: CR events: 5493948
